# Mini-Project 03: Uncertainty-Aware Predictions Pipeline

Combines Bayesian Linear Regression (epistemic uncertainty) and Conformal Prediction (guaranteed coverage).

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust import setup: traverse up until 'src' directory is found
root_dir = Path.cwd().resolve()
while not (root_dir / "src").exists() and root_dir != root_dir.parent:
    root_dir = root_dir.parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.bayesian import BayesianLinearRegression
from src.conformal import ConformalLinearRegression
from src.solvers import ClosedFormLinearRegression

df = pd.read_csv(root_dir / "data" / "california_housing" / "california_housing.csv")
X = df[["MedInc", "HouseAge"]].values[:1000]
y = df["MedHouseVal"].values[:1000]

# 1. Bayesian Epistemic Uncertainty
bayes = BayesianLinearRegression(alpha=1.0, beta=5.0).fit(X, y)
y_mean, y_std = bayes.predict(X[:10], return_std=True)

# 2. Conformal Interval with 90% Guarantee
conformal = ConformalLinearRegression(estimator=ClosedFormLinearRegression(method="svd"), alpha=0.10, cal_size=0.20, random_state=42)
conformal.fit(X, y)
preds, lower, upper = conformal.predict_interval(X[:10])

comparison_table = pd.DataFrame({
    "True y": y[:10],
    "Bayes Mean": y_mean,
    "Epistemic std": y_std,
    "Conformal Low (90%)": lower,
    "Conformal High (90%)": upper
})
print(comparison_table.round(3).to_string(index=False))

 True y  Bayes Mean  Epistemic std  Conformal Low (90%)  Conformal High (90%)
  4.526       3.843          0.449                2.950                 4.742
  3.585       3.737          0.449                2.833                 4.625
  3.521       3.484          0.449                2.595                 4.388
  3.413       2.861          0.448                1.969                 3.761
  3.422       2.167          0.448                1.272                 3.065
  2.697       2.240          0.448                1.346                 3.138
  2.992       2.094          0.447                1.200                 2.992
  2.414       1.886          0.447                0.991                 2.783
  2.267       1.436          0.447                0.533                 2.326
  2.611       2.107          0.447                1.212                 3.004
